In [ ]:
# [Setup]: 
# Had to install minoconda (Windows): https://www.anaconda.com/download/success
# Had to install Microsoft Visual Code, check Desktop development with C++ in install page: https://visualstudio.microsoft.com/visual-cpp-build-tools/
# Had to install Python extension in VSCodium

# Installs necessary packages. Ensure you have followed the steps above before continuing.
# Had to run in Anaconda prompt: 
    # conda create -n rcbplates python=3.11 -y
    # conda activate rcbplates
    # pip install ipympl
    # conda install -c conda-forge astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config -y
    # # pip install daschlab astropy photutils matplotlib numpy scipy pillow pycairo cairo pkg-config astroquery
    # pip install daschlab

# Then, in VSCodium:
    # Ctrl + Shift + P -> Python: Select Interpreter -> Python 3.11 (rcbplates)


In [ ]:
# Cell 1

from IPython.display import HTML, display

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout,
.cell-output, .cell-output-print, .cell-output-stdout {
    color: white !important;
}
.output * { color: white !important; }
.output .ansi-yellow-fg, .output .ansi-yellow-foreground,
.warning, .Warning, .warnings {
    color: #FFD700 !important;
    text-shadow: 0px 0px 5px rgba(255, 215, 0, 0.3);
}
.output pre, .output code { color: white !important; }
</style>
"""))

from daschlab import open_session
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED
import threading
import time
import traceback
import ipywidgets as widgets
import pandas as pd

session_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB"
)
session_dir.mkdir(parents=True, exist_ok=True)

cutout_dir = session_dir / "cutouts"
manifest_path = session_dir / "plate_manifest.csv"

# ---------------------------------------------------------------------------
# Guard: don't let two download threads run at once if you accidentally
# re-run this cell while a previous download is still going.
# ---------------------------------------------------------------------------
if "_active_download_thread" in globals() and _active_download_thread.is_alive():
    print("⚠️  A download is already running in the background from a previous "
          "run of this cell. Click that run's Pause button and wait for "
          "'Paused' before re-running this cell.")
else:
    # -----------------------------------------------------------------------
    # Top bar: pause button + custom progress display, shown first so it
    # stays pinned at the top of this cell's output.
    # -----------------------------------------------------------------------
    pause_button = widgets.Button(
        description="⏸ Pause",
        button_style="warning",
        layout=widgets.Layout(width="120px", height="36px"),
    )
    progress_html = widgets.HTML(value="<i>Setting up session...</i>")

    top_bar = widgets.HBox(
        [pause_button, progress_html],
        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"),
    )

    log_output = widgets.Output(
        layout=widgets.Layout(
            border="1px solid #444", padding="8px",
            max_height="300px", overflow="auto", background_color="#111",
        )
    )

    display(top_bar, log_output)

    pause_event = threading.Event()

    def on_pause_clicked(b):
        pause_event.set()
        pause_button.description = "Pausing..."
        pause_button.disabled = True

    pause_button.on_click(on_pause_clicked)

    def format_eta(seconds):
        if seconds is None or seconds != seconds or seconds < 0:  # NaN/negative guard
            return "calculating..."
        m, s = divmod(int(seconds), 60)
        h, m = divmod(m, 60)
        return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

    def render_progress(n_done, total, start_time, status_word="Downloading", color="#4CAF50"):
        elapsed = time.monotonic() - start_time
        pct = (n_done / total * 100) if total else 0
        rate = (n_done / elapsed) if elapsed > 0 and n_done > 0 else 0
        remaining = ((total - n_done) / rate) if rate > 0 else None

        bar_width = 380
        filled = int(bar_width * pct / 100)

        progress_html.value = f"""
        <div style="font-family: monospace; font-size: 14px; color: white; line-height: 1.6;">
          <div style="display:flex; align-items:center;">
            <div style="width:{bar_width}px; height:18px; background:#333;
                        border-radius:4px; overflow:hidden; margin-right:12px;">
              <div style="width:{filled}px; height:100%; background:{color};
                          transition: width 0.3s;"></div>
            </div>
            <b>{pct:5.1f}%</b>
          </div>
          <div style="margin-top:4px;">
            {status_word} &nbsp;
            <b>{n_done:,} / {total:,}</b> plates &nbsp;|&nbsp;
            <b>{rate:.2f}</b> plates/sec &nbsp;|&nbsp;
            Elapsed <b>{format_eta(elapsed)}</b> &nbsp;|&nbsp;
            ETA <b>{format_eta(remaining)}</b>
          </div>
        </div>
        """

    # -------------------------------------------------------------------
    # Session + manifest setup
    # -------------------------------------------------------------------
    with log_output:
        sess = open_session(str(session_dir))
        sess.select_target("R CrB")
        sess.select_refcat("apass")

        exposures = sess.exposures()
        print(f"Total exposures : {len(exposures)}")

        try:
            exposures_df = exposures.to_pandas()
        except Exception as e:
            print(f"Could not convert exposures table to pandas ({e}); manifest will have fewer columns.")
            exposures_df = pd.DataFrame(index=range(len(exposures)))

        exposures_df.insert(0, "exposure_index", range(len(exposures_df)))

        if manifest_path.exists():
            manifest_df = pd.read_csv(manifest_path)
            print(f"Loaded existing manifest with {len(manifest_df)} rows.")
        else:
            manifest_df = exposures_df.copy()
            manifest_df["status"] = "not_attempted"
            manifest_df["filename"] = ""
            manifest_df["error_message"] = ""
            manifest_df["last_updated"] = ""

        manifest_df = manifest_df.set_index("exposure_index", drop=False)

        # Reconcile against what's actually on disk, in case a previous run
        # ended ungracefully and some rows never got saved as "downloaded".
        existing_files = {f.name for f in cutout_dir.glob("*.fits")} if cutout_dir.exists() else set()
        if existing_files and "filename" in manifest_df.columns:
            on_disk_mask = manifest_df["filename"].apply(
                lambda f: Path(f).name in existing_files if isinstance(f, str) and f else False
            )
            reconciled = (manifest_df["status"] != "downloaded") & on_disk_mask
            if reconciled.any():
                manifest_df.loc[reconciled, "status"] = "downloaded"
                print(f"Reconciled {reconciled.sum()} rows already on disk but not marked downloaded.")

        remaining_indices = manifest_df.index[manifest_df["status"] != "downloaded"].tolist()
        print(f"{len(remaining_indices)} cutouts remaining to download "
              f"(out of {len(manifest_df)} total).")

    total_remaining = max(len(remaining_indices), 1)
    render_progress(0, total_remaining, time.monotonic(), status_word="Starting...")

    # Conservative -- DASCH's docs warn bulk cutout fetching loads their API server.
    MAX_WORKERS = 6

    def fetch_one(i):
        try:
            result = sess.cutout(i)
            if result is None:
                return i, "unavailable", None, "no cutout available (likely unscanned)"
            return i, "downloaded", str(result), None
        except Exception as e:
            return i, "error", None, str(e)

    def save_manifest():
        """Fault-tolerant save -- e.g. won't crash the thread if the CSV is
        currently open in Excel (PermissionError on Windows). Logs a warning
        instead and just tries again next time."""
        try:
            # reset_index(drop=True) strips the 'exposure_index' index so the
            # 'exposure_index' COLUMN is unambiguous to sort by -- this is
            # only for the on-disk copy, manifest_df itself is untouched.
            manifest_df.reset_index(drop=True).sort_values("exposure_index").to_csv(
                manifest_path, index=False
            )
            return True
        except Exception as e:
            with log_output:
                print(f"⚠️  Couldn't save manifest right now ({e}). "
                      f"If it's open in Excel, close it -- will retry automatically.")
            return False

    # Background download loop
    def run_downloads():
        counts = {"downloaded": 0, "unavailable": 0, "error": 0}
        n_done = 0
        start_time = time.monotonic()
        stopped_early = False

        ex = ThreadPoolExecutor(max_workers=MAX_WORKERS)
        futures = {ex.submit(fetch_one, i): i for i in remaining_indices}
        pending = set(futures.keys())

        try:
            while pending:
                if pause_event.is_set():
                    stopped_early = True
                    with log_output:
                        print("\n⏸ Pause requested — no new downloads will start.")
                        print("   A few in-flight downloads may finish in the background; that's fine.")
                    ex.shutdown(wait=False, cancel_futures=True)
                    break

                done, pending = wait(pending, timeout=0.2, return_when=FIRST_COMPLETED)
                for fut in done:
                    i, status, filename, err_msg = fut.result()
                    manifest_df.loc[i, "status"] = status
                    manifest_df.loc[i, "filename"] = filename or ""
                    manifest_df.loc[i, "error_message"] = err_msg or ""
                    manifest_df.loc[i, "last_updated"] = datetime.now().isoformat(timespec="seconds")
                    counts[status] += 1
                    n_done += 1

                if done:
                    render_progress(n_done, len(futures), start_time)
                    save_manifest()
            else:
                ex.shutdown(wait=True)

        except Exception as e:
            # Catches anything unexpected (network errors, widget update
            # failures, etc.) so the button never gets stuck on "Pausing..."
            stopped_early = True
            with log_output:
                print(f"\n❌ Download thread hit an unexpected error: {e}")
                traceback.print_exc()

        finally:
            # This ALWAYS runs -- even on a crash -- so the UI never freezes.
            saved = save_manifest()
            if not saved:
                # one more attempt after a short wait, in case the file just
                # needed to be closed
                time.sleep(1.0)
                save_manifest()

            cutouts = sorted(cutout_dir.glob("*.fits")) if cutout_dir.exists() else []

            final_word = "Paused" if stopped_early else "Done"
            render_progress(n_done, len(futures), start_time,
                             status_word=final_word,
                             color="#FFA726" if stopped_early else "#4CAF50")
            pause_button.description = final_word
            pause_button.disabled = True

            with log_output:
                print("\n========== " + ("PAUSED" if stopped_early else "FINAL") + " SUMMARY ==========")
                print(f"Total exposures                     : {len(manifest_df)}")
                print(f"Total .fits files on disk           : {len(cutouts)}")
                print(f"This run -- downloaded              : {counts['downloaded']}")
                print(f"This run -- unavailable (unscanned) : {counts['unavailable']}")
                print(f"This run -- errored                 : {counts['error']}")
                print("\nAll-time manifest breakdown:")
                print(manifest_df["status"].value_counts().to_string())
                if stopped_early:
                    print("\nSafe to close now. Rerun this cell anytime to pick back up.")
                print(f"\nManifest saved to: {manifest_path}")
                print("=========================================\n")

    _active_download_thread = threading.Thread(target=run_downloads, daemon=True)
    _active_download_thread.start()

Output(layout=Layout(border_bottom='1px solid #444', border_left='1px solid #444', border_right='1px solid #44…

- Querying API ...


- Querying API ...- Querying API ...

- Querying API ...
- Querying API ...
- Querying API ...


- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00021_i01273m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00012_i00936m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00023_i01358m1s0.fits`
- Fetched 1399680 bytes in 6 seconds
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00024_i01457m1s0.fits`
- Querying API ...
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00022_i01319m1s0.fits`
- Querying API ...


- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00026_i01487m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00027_i01520m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00031_i03634m1s0.fits`
- Querying API ...


...

- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00575_ai00912m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00565_ai00858m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00576_ai00918m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00578_ai00926m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00583_ai00940m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\tes

- Fetched 1399680 bytes in 7 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00584_ac03468m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00586_ai00956m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00585_ai00946m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00588_ai00965m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00590_ac03499m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\tes

- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00592_ai00971m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00593_ai00975m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00595_ai00977m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00594_ac03509m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00596_am01865m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\tes

- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00603_ai00997m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00601_ac03525m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00605_ai01004m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 6 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00604_ai01003m0s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00608_ac03543m1s0.fits`
- Querying API ...
- Fetched 1399680 bytes in 5 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\tes

- Fetched 1399680 bytes in 4 seconds
- Saved `C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts\00609_ac03544m0s0.fits`
- Querying API ...


In [ ]:
# Plate cutout visualizer. Shows the cutout, the header, and pixel data for all downloaded plates.

# Cell 2

from pathlib import Path
from io import BytesIO
from astropy.io import fits
from astropy.time import Time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Dark styling for the dropdown + toggles + table + kill any default borders/backgrounds/outlines
display(HTML("""
<style>

.dark-dropdown select {
    background-color: #1e1e1e !important;
    color: white !important;
    border: 1px solid #555 !important;
}

.dark-dropdown select option {
    background-color: #1e1e1e !important;
    color: white !important;
}

.dark-toggle {
    color: white !important;
    background-color: #333 !important;
    border: 1px solid #777 !important;
    font-weight: bold !important;
    font-size: 13px !important;
    padding: 6px 10px !important;
}

.dark-toggle:hover {
    background-color: #4a4a4a !important;
}

/* Applied on top of dark-toggle when a toggle is switched on, for visibility */
.dark-toggle-active {
    background-color: #2e7d32 !important;
    border: 1px solid #66bb6a !important;
}

.dark-toggle-active:hover {
    background-color: #388e3c !important;
}

.dark-output,
.dark-output .output,
.dark-output .widget-output,
.jp-OutputArea-output,
.jp-OutputArea-child,
.cell-output-ipywidget-background,
.output,
.output_wrapper,
.widgetarea,
.jp-RenderedHTMLCommon,
.jupyter-widgets {
    background-color: #111 !important;
    border: none !important;
    box-shadow: none !important;
    outline: none !important;
}

body, .jp-Notebook, .jp-WindowedPanel-outer, .jp-Cell-outputArea {
    background-color: #111 !important;
}

.dark-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 11px;
}

.dark-table th,
.dark-table td {
    border: 1px solid #333;
    padding: 2px 6px;
    text-align: right;
}

.dark-table th {
    background-color: #222;
    color: white;
}

.table-scroll-box {
    max-height: 500px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
}

.header-table {
    border-collapse: collapse;
    color: #ddd;
    background-color: #111;
    font-family: monospace;
    font-size: 12px;
    width: 100%;
}

.header-table th,
.header-table td {
    border: 1px solid #333;
    padding: 4px 10px;
    text-align: left;
}

.header-table th {
    background-color: #222;
    color: white;
}

.header-scroll-box {
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
    background-color: #111;
    margin-bottom: 20px;
}

</style>
"""))

cutout_dir = Path(
    r"C:\Users\dapur\Downloads\Other\Research\rcb_Star_Dust_Survey\test_CNN\data\R_CrB\cutouts"
)

cutouts = sorted(cutout_dir.glob("*.fits"))
print(f"Found {len(cutouts)} cutouts")


# Pull an observation date out of the header (falls back to filename if none found)
def get_plate_date(path):
    try:
        hdr = fits.getheader(path)
    except Exception:
        return path.stem

    for key in ("DATE-OBS", "DATE_OBS", "DATEORIG"):
        if key in hdr and hdr[key]:
            return str(hdr[key])

    for key in ("MJD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="mjd").iso.split(" ")[0]
            except Exception:
                pass

    for key in ("JD",):
        if key in hdr and hdr[key]:
            try:
                return Time(float(hdr[key]), format="jd").iso.split(" ")[0]
            except Exception:
                pass

    return path.stem


# Build dropdown options as "date - filename"
dropdown_options = []

for f in cutouts:
    filename = str(f.name).strip()

    dropdown_options.append(
        (f"{get_plate_date(f)} - {filename}", f)
    )

dropdown = widgets.Select(
    options=dropdown_options,
    rows=12,
    description="",
    layout=widgets.Layout(width="1200px", height="300px")
)

dropdown.add_class("dark-dropdown")

dropdown_container = widgets.HBox(
    [dropdown],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)


# Toggles for header + pixel data
toggle_header = widgets.ToggleButton(
    value=False,
    description="Show Header",
    layout=widgets.Layout(width="160px")
)

toggle_data = widgets.ToggleButton(
    value=False,
    description="Show Pixel Data [SLOW]",
    layout=widgets.Layout(width="180px")
)

toggle_header.add_class("dark-toggle")
toggle_data.add_class("dark-toggle")


def make_label_handler(toggle, label):
    def handler(change):
        if change["new"]:
            toggle.description = f"Hide {label}"
            toggle.add_class("dark-toggle-active")
        else:
            toggle.description = f"Show {label}"
            toggle.remove_class("dark-toggle-active")
    return handler


toggle_header.observe(make_label_handler(toggle_header, "Header"), names="value")
toggle_data.observe(make_label_handler(toggle_data, "Pixel Data"), names="value")


toggles_container = widgets.HBox(
    [toggle_header, toggle_data],
    layout=widgets.Layout(
        justify_content="center",
        width="100%",
        margin="10px 0px"
    )
)


output = widgets.Output(
    layout=widgets.Layout(
        padding="10px",
        width="100%",
        display="flex",
        justify_content="center",
        align_items="center"
    )
)

output.add_class("dark-output")

output_container = widgets.HBox(
    [output],
    layout=widgets.Layout(
        justify_content="center",
        width="100%"
    )
)

headers_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
headers_output.add_class("dark-output")

headers_container = widgets.HBox(
    [headers_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

data_output = widgets.Output(layout=widgets.Layout(padding="10px", width="100%"))
data_output.add_class("dark-output")

data_container = widgets.HBox(
    [data_output],
    layout=widgets.Layout(justify_content="center", width="100%")
)

current_path = None
current_data = None

def render_header():
    with headers_output:
        clear_output(wait=True)

        if not toggle_header.value or current_path is None:
            return

        header = fits.getheader(current_path)

        header_df = pd.DataFrame(list(header.items()), columns=['Keyword', 'Value'])

        header_html = header_df.to_html(
            classes="header-table",
            border=0,
            index=False
        )

        display(HTML("<h3 style='color:white;text-align:center;'>FITS Header</h3>"))
        display(HTML(f'<div class="header-scroll-box">{header_html}</div>'))


def render_data():
    with data_output:
        clear_output(wait=True)

        if not toggle_data.value or current_data is None:
            return

        df = pd.DataFrame(current_data)
        table_html = df.to_html(classes="dark-table", border=0)

        display(HTML("<h3 style='color:white;text-align:center;'>Pixel Data</h3>"))
        display(HTML(f'<div class="table-scroll-box">{table_html}</div>'))


def show_plate(change):
    global current_path, current_data

    current_path = change["new"]
    current_data = fits.getdata(current_path)

    with output:
        clear_output(wait=True)

        fig = plt.figure(figsize=(12, 10), facecolor="#111111")

        gs = gridspec.GridSpec(1, 3, width_ratios=[0.05, 1.0, 0.05], wspace=0.03)

        ax_spacer = fig.add_subplot(gs[0])
        ax = fig.add_subplot(gs[1])
        cax = fig.add_subplot(gs[2])

        ax_spacer.axis("off")

        fig.patch.set_facecolor("#111111")
        ax.set_facecolor("#111111")

        im = ax.imshow(current_data, origin="lower", cmap="gray")

        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label("Plate Intensity", color="white", fontsize=14)
        cbar.ax.tick_params(colors="white")

        ax.set_title(current_path.stem, fontsize=28, color="white", pad=16)
        ax.set_xlabel("X Pixel", color="white")
        ax.set_ylabel("Y Pixel", color="white")
        ax.tick_params(colors="white")

        buf = BytesIO()
        fig.savefig(buf, format="png", facecolor=fig.get_facecolor(), bbox_inches="tight")
        plt.close(fig)
        buf.seek(0)

        display(widgets.Image(value=buf.read(), format="png"))

    render_header()
    render_data()


dropdown.observe(show_plate, names="value")
toggle_header.observe(lambda change: render_header(), names="value")
toggle_data.observe(lambda change: render_data(), names="value")


display(
    dropdown_container,
    toggles_container,
    output_container,
    headers_container,
    data_container
)


# Show first plate automatically
if cutouts:
    show_plate({"new": cutouts[0]})

Found 10629 cutouts
